# Multi-column bars in a `Dag`

A single `Resample` runs one reducer over one value stream. That is enough for a
bar statistic computed from a single series, but real bars are built from
*several* streams at once: open and close come from price, buy and sell volume
come from signed volume, and every column has to agree on where the bars begin
and end.

Inside a graph you place one `Resample` node per column, each on whatever per-tick
expression it needs, rooted at `Input` nodes rather than functors applied to one
array. Aligning them with `CombineLatest` binds them to data only when the graph
is called.

Because every `Resample` shares the same `freq=` and clock, all columns land on a
single bar grid and cannot drift relative to each other.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from screamer import (First, Last, ExpandingMax, ExpandingMin, ExpandingSum,
                      PosPart, NegPart, Resample, CombineLatest)
from screamer.dag import Input, Dag

rng = np.random.default_rng(7)
n = 400
BAR = 40

t_arr     = np.arange(n, dtype=np.int64)
price_arr = 100.0 + np.cumsum(rng.normal(size=n))

# Signed volume: positive is buyer-initiated, negative is seller-initiated.
vol_arr = rng.normal(size=n) * rng.integers(1, 5, size=n)

pd.DataFrame({
    "t":     t_arr[:5],
    "price": price_arr[:5].round(3),
    "vol":   vol_arr[:5].round(2),
}).set_index("t")

## Inputs are named ports, not data

An `Input` is a placeholder for a stream that will be supplied later. Applying a
functor to one does not compute anything: it builds an expression. `First()(price)`
is a node meaning "the first price in each bar", not a number.

This is what lets a reducer sit on top of a per-tick transform. Buy volume is the
running sum of the positive part of signed volume, so the per-tick transform is
`PosPart()(vol)` and the per-bar reducer is `ExpandingSum()`:

```
ExpandingSum()(PosPart()(vol))
```

`PosPart` and `NegPart` are the signed-part decomposition. Every real number
satisfies `x = PosPart(x) - NegPart(x)`, with both parts non-negative: `PosPart`
keeps positive values and zeroes the rest, `NegPart` keeps the magnitude of
negative values and zeroes the rest. Summing each part over a bar splits total
traded volume into buyer- and seller-initiated halves.

In [ ]:
price, vol = Input("price"), Input("vol")

# Nothing is computed here. Each node is a per-bar reducer over an expression.
open_b  = Resample(freq=BAR, agg=First())(price)
high_b  = Resample(freq=BAR, agg=ExpandingMax())(price)
low_b   = Resample(freq=BAR, agg=ExpandingMin())(price)
close_b = Resample(freq=BAR, agg=Last())(price)
buy_b   = Resample(freq=BAR, agg=ExpandingSum())(PosPart()(vol))
sell_b  = Resample(freq=BAR, agg=ExpandingSum())(NegPart()(vol))

# CombineLatest aligns the six bar streams onto one shared grid.
bars = CombineLatest()(open_b, high_b, low_b, close_b, buy_b, sell_b)
COLUMNS = ["open", "high", "low", "close", "buy", "sell"]

print(type(bars).__name__)

## Assembling and calling the graph

`Dag(inputs, outputs)` freezes the expression into a runnable graph. Data binds at
call time: each input is supplied as a `(values, index)` pair, keyed by the name
given to its `Input`. Each stream carries its own index, which drives the bar
boundaries; `price` and `vol` share the same clock here, so their bars align.

The result is a `(values, index)` pair: the aligned six-column bar array and its
shared bar index.

In [ ]:
dag = Dag([price, vol], [bars])

ohlcv, ohlcv_idx = dag(
    price=(price_arr, t_arr),
    vol=(vol_arr, t_arr),
)

print("columns:", COLUMNS)
pd.DataFrame(
    ohlcv.round(2),
    index=pd.Index(ohlcv_idx.astype(int), name="bar_open"),
    columns=COLUMNS,
)

In [ ]:
o, h, l, c = ohlcv[:, 0], ohlcv[:, 1], ohlcv[:, 2], ohlcv[:, 3]
buy, sell  = ohlcv[:, 4], ohlcv[:, 5]

fig = plt.figure(figsize=(10, 6))
gs = GridSpec(2, 1, height_ratios=[3, 1], hspace=0.05)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1], sharex=ax1)

x = np.arange(len(o))
colors = np.where(c >= o, "seagreen", "crimson")
ax1.vlines(x, l, h, color=colors, lw=1)                        # high-low wicks
ax1.bar(x, np.abs(c - o), bottom=np.minimum(o, c),             # open-close bodies
        color=colors, width=0.6)
ax1.set_ylabel("price")
ax1.set_title("OHLC bars with buy and sell volume, one clock")
ax1.tick_params(labelbottom=False)

ax2.bar(x,  buy,  color="seagreen", width=0.6)
ax2.bar(x, -sell, color="crimson",  width=0.6)
ax2.axhline(0, color="k", lw=0.5)
ax2.set_ylabel("volume")
ax2.set_xlabel("bar")

## One clock, no drift

Every column comes from a `Resample` on the same clock and bar width, so they are
finalized at the same boundaries by construction. The check below rebuilds the
price columns through an ordinary eager `Resample` on the same clock and confirms
the graph agrees with it.

The buy and sell columns are checked against the decomposition itself: summing the
positive and negative parts per bar must reproduce them, and their difference is
net order flow.

In [ ]:
# Price columns via the eager path, same clock and bar width.
eager_ohlc = Resample(freq=BAR, agg="ohlc")(price_arr, t_arr)

eager_ohlc_vals = eager_ohlc[0]
for i, name in enumerate(("open", "high", "low", "close")):
    np.testing.assert_allclose(ohlcv[:, i], eager_ohlc_vals[:, i], rtol=1e-12)

# Buy and sell against the signed-part decomposition.
buy_check  = Resample(freq=BAR, agg="sum")(np.asarray(PosPart()(vol_arr)), t_arr)
sell_check = Resample(freq=BAR, agg="sum")(np.asarray(NegPart()(vol_arr)), t_arr)

np.testing.assert_allclose(ohlcv[:, 4], buy_check[0],  rtol=1e-12)
np.testing.assert_allclose(ohlcv[:, 5], sell_check[0], rtol=1e-12)

print("graph columns match the eager path and the signed-part decomposition")

net_flow = ohlcv[:, 4] - ohlcv[:, 5]
pd.Series(net_flow.round(2), name="net_flow",
          index=pd.Index(ohlcv_idx.astype(int), name="bar_open"))

## Extending the bar

The set of columns is open-ended. Any expression whose top node is a per-bar
reducer adds one more `Resample`, and `CombineLatest` joins it on the same clock
as the rest. Adding realized volatility per bar is one more node, not a rebuild.

The graph is a value: define it once, and call it with whatever data you bind at
the time.

In [ ]:
from screamer import ExpandingStd

price2, vol2 = Input("price"), Input("vol")
close2 = Resample(freq=BAR, agg=Last())(price2)
std2   = Resample(freq=BAR, agg=ExpandingStd())(price2)
flow2  = Resample(freq=BAR, agg=ExpandingSum())(vol2)

bars2 = CombineLatest()(close2, std2, flow2)
COLS2 = ["close", "vol", "flow"]

out, out_idx = Dag([price2, vol2], [bars2])(
    price=(price_arr, t_arr),
    vol=(vol_arr, t_arr),
)

print("columns:", COLS2)
pd.DataFrame(
    out.round(3),
    index=pd.Index(out_idx.astype(int), name="bar_open"),
    columns=COLS2,
)